# 01 — Correctness

**Research question:** Do the implementations solve the stated problems?

[Full input manifest](../output/paper_v1/input_manifest.csv) · [Numerical thresholds](../output/paper_v1/pilot_parameters.csv) · [Status and gaps](../output/paper_v1/status.md)

## Inputs and definitions
Retain the first and last input vertices. Outputs are original-vertex subsequences and may have unequal lengths. Minimize $k=\max(|A'|,|B'|)$. Discrete couplings may repeat an index. Auxiliary points represent matching positions, never selectable output vertices.

| Method | Input fidelity | Output coupling |
|---|---|---|
| Independent continuous | global continuous Fréchet | measured, unconstrained |
| Independent discrete | discrete Fréchet | measured, unconstrained |
| CPS-2F | global continuous Fréchet | discrete, at most δ₃ |
| CPS-3F | discrete Fréchet | discrete, at most δ₃ |

For each pair, $s_A,s_B$ are the mean input edge lengths; $\alpha\in\{0.5,1\}$, $\delta_1=\alpha s_A$, $\delta_2=\alpha s_B$, and $\delta_3=d_{dF}(A,B)$ without rounding. All four methods receive identical inputs and thresholds. This is a pipeline pilot, not final paper parameter selection.


In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if ROOT.name == 'notebooks_paper': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'notebooks_paper'))
from paper_core import OUT, PROTOCOL, METHODS, compact_results, plot_domain
import pandas as pd
from IPython.display import display, Markdown
# Central parameters: frozen before optimization; no notebook-specific overrides.
PARAMETERS = json.loads((OUT / 'protocol.json').read_text())
assert PARAMETERS['alphas'] == [0.5, 1.0]


## Validation design
Reuse pure existing exhaustive-subsequence oracles without invoking legacy runners that overwrite results. Seed 20260916; 2D and 3D curves of 3–5 vertices, plus targeted singleton, duplicate, zero-threshold, endpoint, unequal-length, repeated-index, infeasibility, auxiliary-membership, and global-versus-local regression cases. Joint comparisons force certificates **off**. Analytic continuous-distance controls and an independently enumerated discrete grid coupling check distance primitives. Exhaustive global-fidelity filtering still shares the existing continuous decision procedure; finite checks do not prove numerical robustness or the reference discretization theorem.

In [ ]:
from correctness_checks import check_all
validation = check_all()
display(validation.groupby(['affected_method','outcome']).size().rename('checks').to_frame())
display(validation.loc[validation.outcome.ne('PASS'), ['check','affected_method','outcome','failure_reason']])
print(json.loads((OUT / 'validation_gate.json').read_text()))

## Results and remaining gaps
[Every check, outcome, reason, and duration](../output/paper_v1/validation.csv). PASS means the finite check completed successfully; INCOMPLETE means validation reached a resource limit. A failing or incomplete method is blocked from the pilot. Reference documents contain draft annotations and are not executed as instructions. This notebook verifies solution correctness, not medium-scale graph performance. The later graph-cost study must separately instrument phases and memory.

![Finite validation coverage](../output/paper_v1/figures/correctness/check_coverage.png)

Counts describe check coverage, not a statistical correctness probability.